Imports and loading from CSV:

In [ ]:
from bookstats.config import FILTERED_DATA
from bookstats.formatting import clean_headers, clean_whitespace
import pandas as pd

df = pd.read_csv(FILTERED_DATA)

# Basic clean for easier investigation
clean_headers(df)
df["publisher"] = clean_whitespace(df["publisher"])

publishers = df["publisher"]



#### Investigating

In [ ]:
len(publishers)

In [ ]:
publishers.value_counts().head(25)

_Note: Overall, the value counts of all publishers are relatively low? The highest being 317 for Vintage, out of 10k books. Needs to be investigated, could be either a lot of unintentional duplicates, but also nature of the data set (high content of nonfiction, cookbooks, scientific books etc.) or the fact that it's split very granularly into local sections._

Top publishers -> Vintage, Penguin Books, Penguin Classic are essentially the same company, but merging them might not be desirable, for now imprints should be separate, in the future add label of parent company when I have reliable source of info to scrape

In [ ]:
# Testing if there is alternative formatting to most common ones:

publishers[publishers.str.contains("penguin", case=False)].value_counts()

Similar finds: Penguin Books, Penguin Books Ltd., Penguin Books Ltd. (London)
Also: Penguin Classics, Penguin Books (Penguin Classics)

However, the first term catches absolute majority. Alternatives contain usually only 1-2 results - but I can't make this generalisation across other publisher suplicates.

In [ ]:
# Testing if there is alternative formatting to most common ones:

publishers[publishers.str.contains("harper", case=False)].value_counts()

Finds: Harper Perennial (different from HarperCollins? 110), Harper Perennial Modern Classics (26)
Also: HarperCollins (110), HarperCollins Publishers (48)

These are some pretty impactful duplicates on one of the biggest publishers - I think 'Publisher' in name should be relatively safe to strip.

In [ ]:
# Verifying if there are notable publishers where 'publisher' or 'publishers' in name is key:

publishers[publishers.str.contains("publisher", case=False)].value_counts()

In [ ]:
# Looking specifically at publishers containing an 'and' as that may be risk:

publishers[publishers.str.contains(r" and | & ", regex=True)].value_counts().head(25)

Finds: 
- W. W. Norton & Company (42), W.W. Norton & Company (16)
- Faber & Faber (15), Faber and Faber (13)
- Simon & Schuster (44), Simon and Schuster (5)

#### Publisher normalisation key

Many of the replacements and strips are risky, so the safest option moving forward is to introduce a second 'key' column to aid in decision-making. Ultimately, this step cannot be completely automated or handled by script, due to differences between publishers (what is redundant in one publisher name is key in another). The plan is to review especially the most notable ones.

Testing: 
- '&' replaced by 'and'
- 'Ltd.' stripped
- 'Publisher' / 'Publishers' stripped
- One concrete find -> W.W. Norton & Company (16) change to W. W. Norton & Company (42)

In [ ]:
df["publisher_key"] = df["publisher"].str.replace(" & "," and ")

df["publisher_key"].value_counts().head(25)

In [ ]:
df["publisher_key"] = df["publisher_key"].str.replace(r"\bPublishers?\b","", regex=True)
df["publisher_key"] = clean_whitespace(df["publisher_key"])

df["publisher_key"].value_counts().head(25)

In [ ]:
df["publisher_key"] = df["publisher_key"].str.replace(r"\bLtd\.?","", regex=True)
df["publisher_key"] = clean_whitespace(df["publisher_key"])

df["publisher_key"].value_counts().head(25)

In [ ]:
comparison_values = pd.concat([df["publisher"].value_counts(), df["publisher_key"].value_counts()], axis=1)
comparison_unique = df["publisher"].nunique() - df["publisher_key"].nunique()
comparison_rows = (df["publisher"] != df["publisher_key"]).sum()

print(df["publisher"].nunique())
print(comparison_unique)
print(comparison_rows)
print(comparison_values.head(40))


Entity resolution plan: Apply above filters which allow to collapse groups for reference, then use that key column to see which version is most represented within each group. 

In [ ]:
# Checking effect after writing above steps into module:

from bookstats.publishers import publisher_dedup

clean_publishers = publisher_dedup(df["publisher"])

comparison_values = pd.concat([df["publisher"].value_counts(), clean_publishers.value_counts()], axis=1)
comparison_unique = df["publisher"].nunique() - clean_publishers.nunique()
comparison_rows = (df["publisher"] != clean_publishers).sum()

print(df["publisher"].nunique())
print(comparison_unique)
print(comparison_rows)
print(comparison_values.head(40))

Note: A handful of selected publishers will be hand-picked/cleaned as defined in PUBLISHER_OVERRIDES in publishers.py module file. For those, global targeting/replacing would introduce too many errors, so it is best to hand correct them.

#### Ideas for future further dedup:

1. Normalise imprints & publishers, into separate entities. Find a reliable real-world check for this step. 

2. Consider also splitting by country? There is Penguin UK and Penguin US for example.

3. Run names against fuzzy finding search (like an online ISBN registry?), so that it is precise and differences such as 'W. W. Norton & Company' and 'W.W. Norton & Company' gets cleaned without complex global script.

**Future reference: ISBN Registry**

https://grp.isbn-international.org/search/piid_solr?keys=samhain+publishing